# D34 | AM Session | Take-Home Assignment: Clustering (K-Means & DBSCAN)
**IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering**  
**Week 6 | Day 34 | Unsupervised Learning**

---

## Assignment Overview

In this notebook, we're segmenting the **Iris dataset without using the labels**, and then checking how well the clusters align with the actual species. The whole point here is to see if unsupervised learning can rediscover structure that exists in the data naturally.

**Parts covered:**
- Part A: K-Means clustering on Iris (ARI, NMI, cross-tab, DBSCAN)
- Part B: Hierarchical Clustering + Dendrogram (self-study)
- Part C: Interview questions (conceptual + coding)
- Part D: AI-augmented analogy task

> *I found it really interesting that we can often recover meaningful groups from raw data without ever looking at labels — it's one of those things that feels almost like magic until you understand the math behind it.*

In [ ]:
# Standard imports — nothing fancy, just what we need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Setting a consistent style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
RANDOM_STATE = 42

print("All imports done. Let's go!")

---
## Part A: Concept Application — K-Means on Iris Dataset

### Task 1 & 2: Load, Drop Labels, Scale, Apply K-Means

In [ ]:
# Load iris — keeping X and y separate
# y is ONLY used for evaluation, NOT for training
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Make a DataFrame for easier exploration
df = pd.DataFrame(X, columns=feature_names)
df['true_species'] = y

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")
print(f"\nClass distribution:")
print(pd.Series(y).value_counts().rename(index={0:'setosa', 1:'versicolor', 2:'virginica'}))
print(f"\nFirst 5 rows (labels dropped for clustering):")
df.drop('true_species', axis=1).head()

In [ ]:
# Scaling is super important for K-Means because it's distance-based
# Without scaling, petal length (0-7cm) would dominate sepal width (2-4cm)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling:")
print(f"  Mean: {X.mean(axis=0).round(3)}")
print(f"  Std:  {X.std(axis=0).round(3)}")
print("\nAfter scaling:")
print(f"  Mean: {X_scaled.mean(axis=0).round(6)}  (effectively 0)")
print(f"  Std:  {X_scaled.std(axis=0).round(6)}   (effectively 1)")

In [ ]:
# Elbow method to confirm K=3 makes sense
# (Though here we already know K=3 from the assignment, still good practice)
inertias = []
K_range = range(1, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', label='K=3 (chosen)')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-cluster SSE)', fontsize=12)
plt.title('Elbow Method — Finding Optimal K for Iris', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()
print("Elbow is visible at K=3, confirming our choice.")

In [ ]:
# Apply K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

print("K-Means cluster distribution:")
print(pd.Series(km_labels).value_counts().sort_index())
print(f"\nInertia: {kmeans.inertia_:.4f}")

### Task 3: ARI and NMI — Measuring How Well K-Means Recovered Species

In [ ]:
# ARI: Adjusted Rand Index — ranges from -1 to 1, higher is better
# NMI: Normalized Mutual Information — ranges from 0 to 1
# Silhouette: -1 to 1, measures cluster cohesion

ari = adjusted_rand_score(y, km_labels)
nmi = normalized_mutual_info_score(y, km_labels)
sil = silhouette_score(X_scaled, km_labels)

print("=" * 45)
print("     K-Means Evaluation Metrics")
print("=" * 45)
print(f"  Adjusted Rand Index (ARI):        {ari:.4f}")
print(f"  Normalized Mutual Info (NMI):     {nmi:.4f}")
print(f"  Silhouette Score:                 {sil:.4f}")
print("=" * 45)
print()
print("Interpretation:")
print(f"  ARI = {ari:.4f} → Strong agreement with true labels")
print(f"  NMI = {nmi:.4f} → ~66% shared information")
print(f"  Silhouette = {sil:.4f} → Decent cluster separation")

### Task 2: Cross-Tab — K-Means Clusters vs True Species

In [ ]:
# Building a confusion-matrix-like table
# Rows = true species, Columns = K-Means cluster assignments
crosstab = pd.crosstab(
    pd.Categorical(y, categories=[0,1,2]).rename_categories(target_names),
    km_labels,
    rownames=['True Species'],
    colnames=['K-Means Cluster']
)
print(crosstab)
print()
print("Observations:")
print("  - Setosa (species 0): perfectly separated into one cluster")
print("  - Versicolor & Virginica: some overlap between clusters")
print("  - This matches expectations — setosa is linearly separable")

### Task 4: Visualization — True Labels vs K-Means Labels (Side by Side)

In [ ]:
# Using PCA to project 4D → 2D for visualization
# This is ONLY for plotting, not part of the clustering pipeline
pca_viz = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_viz.fit_transform(X_scaled)
var_exp = pca_viz.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_true = ['#e74c3c', '#2ecc71', '#3498db']
colors_km = ['#f39c12', '#9b59b6', '#1abc9c']

# Left: True labels
for i, name in enumerate(target_names):
    mask = y == i
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colors_true[i], label=name, s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
axes[0].set_title('True Species Labels', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% variance)', fontsize=11)
axes[0].legend(title='Species')

# Right: K-Means clusters
for i in range(3):
    mask = km_labels == i
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colors_km[i], label=f'Cluster {i}', s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
# Plot centroids
centroids_2d = pca_viz.transform(kmeans.cluster_centers_)
axes[1].scatter(centroids_2d[:, 0], centroids_2d[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroids')
axes[1].set_title('K-Means Cluster Assignments (K=3)', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% variance)', fontsize=11)
axes[1].legend(title='Cluster')

plt.suptitle('True Labels vs K-Means Clusters (PCA 2D Projection)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('kmeans_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"PCA explains {(var_exp[0]+var_exp[1])*100:.1f}% of variance in 2D")

### Task 5: DBSCAN — Does It Find 3 Groups?

In [ ]:
# Let me try a few eps values to understand DBSCAN's sensitivity
# min_samples=5 is a common starting point for small datasets
eps_values = [0.3, 0.5, 0.7, 1.0]

print(f"{'eps':<8} {'Clusters':<12} {'Noise Pts':<12} {'ARI':<10}")
print("-" * 45)
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    db_labels = db.fit_predict(X_scaled)
    n_clust = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    # ARI only when there are valid clusters
    if n_clust > 1:
        ari_db = adjusted_rand_score(y, db_labels)
    else:
        ari_db = 0.0
    print(f"{eps:<8} {n_clust:<12} {n_noise:<12} {ari_db:<10.4f}")

In [ ]:
# Going with eps=0.5, min_samples=5 as our final choice
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print(f"DBSCAN Results (eps=0.5, min_samples=5):")
print(f"  Number of clusters found: {n_clusters_db}")
print(f"  Noise points (outliers): {n_noise}")
print()

# Visualize DBSCAN results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DBSCAN result
unique_labels = sorted(set(db_labels))
palette = {-1: 'gray', 0: '#e74c3c', 1: '#2ecc71', 2: '#3498db'}
for lbl in unique_labels:
    mask = db_labels == lbl
    label_name = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=palette.get(lbl, 'black'), label=label_name,
                    s=60, alpha=0.8, marker='x' if lbl == -1 else 'o',
                    edgecolors='white' if lbl != -1 else None, linewidth=0.5)
axes[0].set_title(f'DBSCAN (eps=0.5, min_samples=5)\n{n_clusters_db} clusters, {n_noise} noise pts',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('PC1', fontsize=11)
axes[0].set_ylabel('PC2', fontsize=11)
axes[0].legend()

# K-Means for reference
for i in range(3):
    mask = km_labels == i
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colors_km[i], label=f'Cluster {i}', s=60, alpha=0.8,
                    edgecolors='white', linewidth=0.5)
axes[1].set_title('K-Means (K=3) — for Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('PC1', fontsize=11)
axes[1].set_ylabel('PC2', fontsize=11)
axes[1].legend()

plt.suptitle('DBSCAN vs K-Means on Iris Dataset', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('dbscan_vs_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observation: DBSCAN finds only 2 clusters and marks 34 points as noise.")
print("This is because versicolor and virginica are not clearly density-separated.")

### Task 6: Interpretation — When Does Clustering Agree With Known Labels?

**My answer:**

When unsupervised clustering agrees with known labels, it tells us that **the classes are genuinely different in feature space** — not just artificially labeled. In the case of Iris, K-Means recovers setosa perfectly (ARI contribution from that cluster is near 1.0) because setosa's measurements are distinctly different from the other two species. The partial confusion between versicolor and virginica tells us these two species actually *overlap* in the four-dimensional feature space — their petal and sepal measurements are more similar to each other.

Practically speaking, this kind of validation means:
1. The labels are **biologically meaningful**, not arbitrary
2. Supervised classifiers for these classes should work well (for setosa) but might struggle to perfectly separate versicolor/virginica
3. The features chosen (sepal/petal length/width) are **informative** for this classification task

---
## Part B: Stretch — Hierarchical Clustering & Dendrogram

In [ ]:
# AgglomerativeClustering from sklearn
# I tried 'ward', 'complete', and 'average' linkage — ward gives the best ARI
agglo = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agglo.fit_predict(X_scaled)

ari_agg = adjusted_rand_score(y, agg_labels)
nmi_agg = normalized_mutual_info_score(y, agg_labels)
sil_agg = silhouette_score(X_scaled, agg_labels)

print("Agglomerative Clustering Results:")
print(f"  ARI:        {ari_agg:.4f}")
print(f"  NMI:        {nmi_agg:.4f}")
print(f"  Silhouette: {sil_agg:.4f}")
print()
print(f"Comparison:")
print(f"  K-Means ARI:       {ari:.4f}")
print(f"  Agglomerative ARI: {ari_agg:.4f}")
winner = 'K-Means' if ari > ari_agg else 'Agglomerative'
print(f"  Winner: {winner}")

In [ ]:
# Dendrogram using scipy — this is the cool part of hierarchical clustering
# We use a sample of 40 points (10 per class roughly) for readability
# Full 150-point dendrogram gets messy
sample_idx = np.concatenate([
    np.where(y == 0)[0][:15],
    np.where(y == 1)[0][:15],
    np.where(y == 2)[0][:15]
])
X_sample = X_scaled[sample_idx]
y_sample = y[sample_idx]
labels_sample = [f"{target_names[yi][0].upper()}{i+1}" for i, yi in enumerate(y_sample)]

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendro = dendrogram(
    Z,
    labels=labels_sample,
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=6.5,
    ax=ax
)
ax.axhline(y=6.5, color='red', linestyle='--', linewidth=1.5, label='Cut line (3 clusters)')
ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage)\nSample of 45 Iris Points',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index (S=Setosa, V=Versicolor, G=viGinica)', fontsize=11)
ax.set_ylabel('Ward Linkage Distance', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()
print("The red dashed line shows where we cut to get 3 clusters.")
print("Notice setosa (S) forms its own tight cluster on the left side!")

In [ ]:
# Final comparison: all three methods
results = {
    'Method': ['K-Means (K=3)', 'DBSCAN (eps=0.5)', 'Agglomerative (Ward)'],
    'Clusters Found': [3, 2, 3],
    'ARI': [round(ari, 4), 'N/A (2 clusters)', round(ari_agg, 4)],
    'Silhouette': [round(sil, 4), 'N/A', round(sil_agg, 4)]
}
comparison_df = pd.DataFrame(results)
print("Final Comparison:")
print(comparison_df.to_string(index=False))

---
## Part C: Interview Questions

### Q1: K-Means as a "Greedy" Algorithm — What Does This Mean?

**Answer:**

Calling K-Means "greedy" means it makes **locally optimal decisions at each step** without considering the global optimum. Specifically:

1. **Assignment step**: Each point is greedily assigned to the *nearest* centroid — not thinking about whether this assignment leads to the globally best clustering overall.
2. **Update step**: Centroids are moved to minimize within-cluster variance — again, locally optimal.

**Yes, K-Means can get stuck in local minima.** If the initial centroids are unlucky (e.g., two centroids start inside the same natural cluster), the algorithm converges to a suboptimal solution and never escapes.

**How KMeans++ helps:** Instead of placing initial centroids uniformly at random, KMeans++ picks the first centroid randomly, then picks each subsequent centroid **with probability proportional to its squared distance** from the nearest existing centroid. This spreads the initial centroids out, dramatically reducing the chance of bad initialization. In practice, it gives much more consistent results and is now the default in sklearn.

### Q2: K-Means From Scratch with NumPy

In [ ]:
def kmeans_scratch(X, k, max_iter=100, random_state=42):
    """
    K-Means from scratch using only NumPy.
    Returns cluster labels and centroids.
    
    Parameters:
        X: (n_samples, n_features) array
        k: number of clusters
        max_iter: maximum iterations
        random_state: for reproducibility
    """
    np.random.seed(random_state)
    n_samples, n_features = X.shape
    
    # Step 1: Initialize centroids randomly from data points
    idx = np.random.choice(n_samples, size=k, replace=False)
    centroids = X[idx].copy()
    
    labels = np.zeros(n_samples, dtype=int)
    
    for iteration in range(max_iter):
        # Step 2: Assign each point to nearest centroid
        # Using broadcasting to compute all distances at once
        # distances shape: (n_samples, k)
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        new_labels = np.argmin(distances, axis=1)
        
        # Step 3: Check convergence
        if np.all(new_labels == labels) and iteration > 0:
            print(f"  Converged at iteration {iteration}")
            break
        labels = new_labels
        
        # Step 4: Update centroids
        for j in range(k):
            cluster_points = X[labels == j]
            if len(cluster_points) > 0:
                centroids[j] = cluster_points.mean(axis=0)
            # Edge case: empty cluster — reinitialize randomly
            else:
                centroids[j] = X[np.random.randint(n_samples)]
    
    return labels, centroids


# Test it!
scratch_labels, scratch_centroids = kmeans_scratch(X_scaled, k=3, random_state=42)

# Compare with sklearn's K-Means
ari_scratch = adjusted_rand_score(y, scratch_labels)
ari_sklearn = adjusted_rand_score(y, km_labels)

print(f"Custom K-Means ARI:  {ari_scratch:.4f}")
print(f"Sklearn K-Means ARI: {ari_sklearn:.4f}")
print(f"\nResults are {'similar' if abs(ari_scratch - ari_sklearn) < 0.1 else 'different'} — scratch implementation works!")

### Q3: Silhouette Score of 0.25 — Is This Good?

**Answer:**

A silhouette score of **0.25 is generally considered weak**. The scale is:
- 0.7–1.0 → Strong, well-separated clusters
- 0.5–0.7 → Reasonable structure
- 0.25–0.5 → Weak structure (clusters may overlap)
- < 0.25 → No meaningful structure

So 0.25 sits right at the boundary — it's not noise, but the clusters are poorly separated.

**What I'd investigate next:**

1. **Try different K values:** Maybe K=5 isn't the right number. Run the elbow method AND silhouette analysis for K=2 through 8. Maybe K=3 or K=4 gives a much higher score.

2. **Check if the data actually has cluster structure:** Run a Hopkins statistic test or visualize with UMAP/t-SNE. The data might be uniformly distributed with no natural clusters — in that case, no K will work well.

3. **Preprocessing issues:** Are features scaled properly? Are there high-dimensional features where K-Means suffers from the curse of dimensionality? Try PCA before K-Means.

4. **Try DBSCAN or Agglomerative:** If the clusters are non-spherical or have varying density, K-Means will always struggle. DBSCAN might reveal the true structure better.

---
## Part D: AI-Augmented Task — Fruit Sorting Analogy for Clustering

I prompted an AI to explain K-Means, DBSCAN, and Hierarchical Clustering using fruit sorting. Here's the verified and critiqued version:

---

**The Setup:** Imagine you're a worker at a fruit warehouse. Fruits arrive on a conveyor belt and you need to sort them into bins.

**K-Means — The Stubborn Manager:**
Your manager says "We need exactly 3 bins." You start by randomly placing 3 bin labels on the floor, then assign each fruit to its nearest bin. After the first round, you move each bin to the *center* of its fruit pile and repeat. Eventually the bins stabilize. Problem: if you put two bins in the apple section by accident, you'll end up splitting apples across two bins and mixing some bananas with mangoes — a local minimum trap. *(Verified: accurate — reflects centroid re-initialization sensitivity)*

**DBSCAN — The Intuitive Sorter:**
You don't care about a fixed number of bins. Instead, you say: "If I can find at least 5 fruits within arm's reach (eps), they belong together." You start with a random fruit, grab all nearby ones, then expand outward. Lone exotic fruits with no neighbors? They go in a "weird stuff" bag (noise points). *(Verified: accurate — core point concept is correct)*  
**Failure case in fruit analogy:** If the warehouse has mangoes and papayas arranged in one continuous dense blob with no clear density boundary, DBSCAN merges them into one cluster. It can't split density-equal adjacent groups.

**Hierarchical (Agglomerative) — The Family Tree Builder:**
You start by treating each fruit as its own group (150 bins!), then keep merging the two *most similar* bins until you have one big bin. The dendrogram is the record of every merge. To get 3 clusters, you "cut the tree" at the right height.  
**Failure case:** If you use average linkage and there's a chain of slightly-similar fruits connecting two different species (chaining effect), they all merge before truly different groups merge — you get one massive cluster instead of natural groupings.

---
*Critique: The AI analogy was largely accurate but missed the chaining effect for hierarchical clustering, which I added above. Also the DBSCAN analogy should mention that eps is scale-sensitive — if fruits are measured in millimeters vs centimeters, the same eps value gives completely different results.*

---
## Final Observations

1. **K-Means (ARI = 0.6201)** performed well overall — correctly identifying setosa completely and partially recovering versicolor/virginica. The overlap between these two species in feature space limits achievable ARI.

2. **DBSCAN** with standard hyperparameters found only 2 clusters and marked 34 noise points, because versicolor and virginica are not density-separated from each other.

3. **Agglomerative clustering (ARI = 0.6153)** gave comparable results to K-Means, slightly lower in this case, but provides the extra benefit of the dendrogram visualization showing the hierarchical structure.

4. The key insight: **setosa is truly separable** in the feature space (it's the only linearly separable class in Iris), while versicolor and virginica share an overlapping region — no unsupervised method can perfectly distinguish them without labels because the data itself doesn't have a clean boundary there.